# Data Preprocessing & Feature Engineering

Pipeline hoàn chỉnh: data cleaning, temporal train/test split, categorical encoding, feature engineering, SMOTE variant comparison, và export.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
import lightgbm as lgb
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTEENN

DATA_RAW = 'D:/AI66A_group_10_Topic_7/src/data/raw/bs140513_032310.csv'
DATA_OUT = 'D:/AI66A_group_10_Topic_7/data/processed'
os.makedirs(DATA_OUT, exist_ok=True)

## Bước 1 — Data Cleaning

In [ ]:
df = pd.read_csv(DATA_RAW)

# Strip whitespace và quotes từ string columns
str_cols = ['customer', 'age', 'gender', 'zipcodeOri', 'merchant', 'zipMerchant', 'category']
for c in str_cols:
    df[c] = df[c].str.strip().str.strip("'")

# Flag age unknown trước khi map
df['age_unknown'] = (df['age'] == 'U').astype(int)

# Map 'U' -> '-1', cast to int
df['age'] = df['age'].replace('U', '-1').astype(int)

# Drop zero-variance columns
df.drop(columns=['zipcodeOri', 'zipMerchant'], inplace=True)

In [ ]:
print('Dtypes:')
print(df.dtypes)
print('\nMissing values:', df.isnull().sum().sum())
print('age_unknown counts:', df['age_unknown'].value_counts().to_dict())
print('age unique values:', sorted(df['age'].unique()))

## Bước 2 — Temporal Train/Test Split

Sort theo `step` tăng dần, lấy 80% đầu làm train, 20% cuối làm test.

In [ ]:
df = df.sort_values('step').reset_index(drop=True)
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].reset_index(drop=True)
test_df  = df.iloc[split_idx:].reset_index(drop=True)

print(f'Train: {train_df.shape}  |  fraud rate: {train_df["fraud"].mean():.4%}')
print(f'Test:  {test_df.shape}   |  fraud rate: {test_df["fraud"].mean():.4%}')

## Bước 3 — Categorical Encoding

Fit `LabelEncoder` trên train, transform cả train và test. Unseen labels trong test → -1.

In [ ]:
cat_cols = ['gender', 'category', 'merchant', 'customer']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    le.fit(train_df[col])
    classes_map = {v: i for i, v in enumerate(le.classes_)}

    train_df[col] = train_df[col].map(classes_map)
    test_df[col]  = test_df[col].map(classes_map).fillna(-1).astype(int)
    encoders[col] = le

print('Encoding done.')
for col in cat_cols:
    unseen = (test_df[col] == -1).sum()
    print(f'  {col}: {train_df[col].nunique()} train classes, {unseen} unseen in test')

## Bước 4 — Feature Engineering

Tất cả aggregation và risk encoding được tính từ `ref_df` (= `train_df`) để tránh data leakage.

In [ ]:
def add_features(df, ref_df):
    df = df.copy()
    train_mean = ref_df['amount'].mean()
    train_std  = ref_df['amount'].std()

    # --- Amount features ---
    df['amount_log']    = np.log1p(df['amount'])
    df['amount_zscore'] = (df['amount'] - train_mean) / (train_std + 1e-8)

    # --- Customer behavioral aggregation (from ref_df) ---
    cust_stats = (
        ref_df.groupby('customer')['amount']
        .agg(cust_mean='mean', cust_std='std', cust_max='max', cust_tx_count='count')
        .reset_index()
    )
    cust_stats['cust_std'] = cust_stats['cust_std'].fillna(0)

    df = df.merge(cust_stats, on='customer', how='left')
    df['cust_mean']     = df['cust_mean'].fillna(train_mean)
    df['cust_std']      = df['cust_std'].fillna(train_std)
    df['cust_max']      = df['cust_max'].fillna(df['amount'])
    df['cust_tx_count'] = df['cust_tx_count'].fillna(1).astype(int)

    df['amount_deviation']    = (df['amount'] - df['cust_mean']) / (df['cust_std'] + 1e-8)
    df['amount_to_max_ratio'] = df['amount'] / (df['cust_max'] + 1e-8)

    # --- Risk encoding (fit on ref_df only) ---
    global_fraud_rate = ref_df['fraud'].mean()
    cat_risk   = ref_df.groupby('category')['fraud'].mean()
    merch_risk = ref_df.groupby('merchant')['fraud'].mean()

    df['category_risk'] = df['category'].map(cat_risk).fillna(global_fraud_rate)
    df['merchant_risk'] = df['merchant'].map(merch_risk).fillna(global_fraud_rate)

    # --- Behavioral flags (chronological order) ---
    df = df.sort_values('step').reset_index(drop=True)
    df['is_new_merchant'] = (
        ~df.groupby('customer')['merchant'].transform(lambda x: x.duplicated())
    ).astype(int)
    df['is_new_category'] = (
        ~df.groupby('customer')['category'].transform(lambda x: x.duplicated())
    ).astype(int)
    df['is_round_amount'] = (df['amount'] % 10 == 0).astype(int)
    df['is_micro_tx']     = (df['amount'] < 1).astype(int)

    # --- Time features ---
    df['hour']       = df['step'] % 24
    df['is_night']   = ((df['hour'] >= 23) | (df['hour'] <= 5)).astype(int)
    df['is_weekend'] = (((df['step'] // 24) % 7) >= 5).astype(int)

    # --- Demographic deviation ---
    age_stats = (
        ref_df.groupby('age')['amount']
        .agg(age_group_mean='mean', age_group_std='std')
        .reset_index()
    )
    age_stats['age_group_std'] = age_stats['age_group_std'].fillna(0)

    df = df.merge(age_stats, on='age', how='left')
    df['age_group_mean'] = df['age_group_mean'].fillna(train_mean)
    df['age_group_std']  = df['age_group_std'].fillna(train_std)
    df['age_group_deviation'] = (
        (df['amount'] - df['age_group_mean']) / (df['age_group_std'] + 1e-8)
    )

    return df

In [ ]:
train_df_fe = add_features(train_df, ref_df=train_df)
test_df_fe  = add_features(test_df,  ref_df=train_df)

print(f'Train shape: {train_df_fe.shape}  |  Test shape: {test_df_fe.shape}')
print('NaN in train:', train_df_fe.isnull().sum().sum())
print('NaN in test: ', test_df_fe.isnull().sum().sum())

## Bước 5 — SMOTE Variant Comparison

So sánh 4 cấu hình resampling bằng LightGBM proxy với 3-fold stratified CV (shuffle=False). Metric: average_precision (AUC-PR). Lưu ý: SMOTEENN có thể chạy lâu trên dataset lớn.

In [ ]:
feature_cols = [c for c in train_df_fe.columns if c != 'fraud']
X_tr_all = train_df_fe[feature_cols].values
y_tr_all  = train_df_fe['fraud'].values

resamplers = {
    'No resampling':        None,
    'SMOTE(0.1)':           SMOTE(sampling_strategy=0.1, random_state=42),
    'BorderlineSMOTE(0.1)': BorderlineSMOTE(sampling_strategy=0.1, random_state=42),
    'SMOTEENN(0.1)':        SMOTEENN(sampling_strategy=0.1, random_state=42),
}

lgb_params = dict(
    n_estimators=200, learning_rate=0.05, num_leaves=31,
    random_state=42, verbose=-1, n_jobs=-1
)

cv   = StratifiedKFold(n_splits=3, shuffle=False)
rows = []

for name, resampler in resamplers.items():
    print(f'Running: {name}...')
    ap_scores, sizes = [], []
    for tr_idx, val_idx in cv.split(X_tr_all, y_tr_all):
        X_cv, y_cv   = X_tr_all[tr_idx], y_tr_all[tr_idx]
        X_val, y_val = X_tr_all[val_idx], y_tr_all[val_idx]
        if resampler is not None:
            X_cv, y_cv = resampler.fit_resample(X_cv, y_cv)
        clf = lgb.LGBMClassifier(**lgb_params)
        clf.fit(X_cv, y_cv)
        ap_scores.append(average_precision_score(y_val, clf.predict_proba(X_val)[:, 1]))
        sizes.append(len(X_cv))
    rows.append({
        'Variant':        name,
        'Mean AUC-PR':    round(np.mean(ap_scores), 4),
        'Std':            round(np.std(ap_scores), 4),
        'Avg Train Size': int(np.mean(sizes)),
    })

results_df   = pd.DataFrame(rows)
best_variant = results_df.loc[results_df['Mean AUC-PR'].idxmax(), 'Variant']

print()
print(results_df.to_string(index=False))
print(f'\nBest variant: {best_variant}')

In [ ]:
X_train = train_df_fe[feature_cols].values
y_train = train_df_fe['fraud'].values
X_test  = test_df_fe[feature_cols].values
y_test  = test_df_fe['fraud'].values

resampler_map = {
    'SMOTE(0.1)':           SMOTE(sampling_strategy=0.1, random_state=42),
    'BorderlineSMOTE(0.1)': BorderlineSMOTE(sampling_strategy=0.1, random_state=42),
    'SMOTEENN(0.1)':        SMOTEENN(sampling_strategy=0.1, random_state=42),
}

best_resampler = resampler_map.get(best_variant)
if best_resampler is not None:
    X_train_res, y_train_res = best_resampler.fit_resample(X_train, y_train)
else:
    X_train_res, y_train_res = X_train.copy(), y_train.copy()

print(f'X_train_res: {X_train_res.shape}  |  fraud rate: {y_train_res.mean():.4%}')

## Bước 6 — Export

Lưu tất cả artifacts vào `data/processed/` để dùng trong training notebook.

In [ ]:
export_map = {
    'X_train':      X_train,
    'X_test':       X_test,
    'y_train':      y_train,
    'y_test':       y_test,
    'X_train_res':  X_train_res,
    'y_train_res':  y_train_res,
    'encoders':     encoders,
    'feature_cols': feature_cols,
}

for name, obj in export_map.items():
    joblib.dump(obj, os.path.join(DATA_OUT, f'{name}.pkl'))

print('Saved to:', DATA_OUT)
for name in export_map:
    path = os.path.join(DATA_OUT, f'{name}.pkl')
    print(f'  {name}.pkl  ({os.path.getsize(path)/1024:.0f} KB)')

In [ ]:
print('=== Feature List ===')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

print(f'\nX_train_res shape: {X_train_res.shape}')
print(f'X_test shape:      {X_test.shape}')